# SW-11-CSharp-KnowledgeGraphs

**Twin C# (dotNetRDF) du notebook Python [SW-11-Python-KnowledgeGraphs](SW-11-Python-KnowledgeGraphs.ipynb)** (rdflib + kglab + networkx + owlready2).

Ce notebook construit, interroge et valide un **Knowledge Graph** (graphe de connaissances) en C# avec la librairie **dotNetRDF** — l'équivalent .NET de rdflib. Il couvre l'arc pédagogique complet : définition d'un vocabulaire RDF, transformation de données structurées en triplets, requêtage SPARQL, parcours/adjacence, visualisation, et métriques de qualité.

> **Parité .NET ⇄ Python** : marathon #4956. Les concepts (triplet RDF, namespace, SPARQL, graphe d'adjacence, métriques de qualité) sont repris from-scratch en C#. Les sorties numériques sont concordantes avec le twin Python.

## 0. Installation de dotNetRDF

dotNetRDF (`VDS.RDF`) est la librairie de référence .NET pour manipuler RDF/SPARQL. C'est l'équivalent direct de `rdflib` en Python.

In [1]:
#r "nuget: dotNetRDF, 3.4.1"
using VDS.RDF;
using VDS.RDF.Parsing;
using VDS.RDF.Query;
using VDS.RDF.Writing.Formatting;
using System;
using System.Collections.Generic;
using System.Linq;

const string XSD = "http://www.w3.org/2001/XMLSchema#";

// Helper statique : cree un noeud URI depuis un QName (persiste entre cellules)
static IUriNode U(IGraph g, string qname) => g.CreateUriNode(qname);
static ILiteralNode LitInt(IGraph g, int v) => g.CreateLiteralNode(v.ToString(), new Uri(XSD + "integer"));

Console.WriteLine("dotNetRDF charge.");

Installed Packages dotNetRDF, 3.4.1

dotNetRDF charge.


## 1. Qu'est-ce qu'un Knowledge Graph ?

Un **Knowledge Graph** (KG) est une représentation structurée de connaissances sous forme d'un graphe orienté étiqueté, où :

- les **nœuds** sont des entités (personnes, films, lieux, concepts),
- les **arêtes** sont des relations typées (`directedBy`, `studiedBy`, `rdf:type`),
- le tout est encodé en **triplets RDF** : `(sujet, prédicat, objet)`.

Contrairement à une base de données relationnelle (schéma figé, tables), un KG est **schema-less** et **extensible** : on peut ajouter de nouveaux types d'entités et de relations sans casser les données existantes. C'est ce qui rend les KG (DBpedia, Wikidata, Google Knowledge Graph) si flexibles pour fusionner des sources hétérogènes.

| Aspect | BDD relationnelle | Graphe RDF / KG |
|--------|-------------------|-----------------|
| Schéma | Figé (tables, colonnes) | Flexible (schema-less) |
| Requête | SQL jointures | SPARQL motifs graphe |
| Fusion sources | Lourde (ETL) | Naturelle (URI partagées) |
| Inférence | Non | Oui (RDFS/OWL reasoners) |

## 2. Construire un Knowledge Graph à partir de données structurées

### 2.1 Définition du vocabulaire RDF

On définit un **namespace** (espace de noms) pour nos URI, puis les nœuds et relations. Prenons un domaine **cinéma** : films, réalisateurs, genres, années.

In [2]:
var g = new Graph();
g.NamespaceMap.AddNamespace("ex", new Uri("http://example.org/cinema#"));
g.NamespaceMap.AddNamespace("rdf", new Uri("http://www.w3.org/1999/02/22-rdf-syntax-ns#"));
g.NamespaceMap.AddNamespace("rdfs", new Uri("http://www.w3.org/2000/01/rdf-schema#"));

IUriNode RDF_TYPE   = U(g, "rdf:type");
IUriNode DIRECTED_BY = U(g, "ex:directedBy");
IUriNode RELEASED   = U(g, "ex:releasedIn");
IUriNode HAS_GENRE  = U(g, "ex:hasGenre");

// Classes
foreach (var d in new[]{"Director","Film","Genre"})
    g.Assert(U(g, "ex:"+d), RDF_TYPE, U(g, "ex:Class"));

Console.WriteLine("Vocabulaire cinema defini.");

Vocabulaire cinema defini.


### 2.2 Transformation de données structurées en triplets

On part d'une table de films (titre, réalisateur, année, genres) — équivalent d'un CSV — et on la transforme en triplets RDF.

In [3]:
// Donnees source : (titre, realisateur, annee, genres) -- equivalent d'un DataFrame Python
var films = new[] {
    (Title:"Inception",       Director:"Nolan",     Year:2010, Genres:new[]{"SciFi","Action"}),
    (Title:"Interstellar",    Director:"Nolan",     Year:2014, Genres:new[]{"SciFi","Drama"}),
    (Title:"Dunkirk",         Director:"Nolan",     Year:2017, Genres:new[]{"Drama","War"}),
    (Title:"PulpFiction",     Director:"Tarantino", Year:1994, Genres:new[]{"Crime","Drama"}),
    (Title:"Django",          Director:"Tarantino", Year:2012, Genres:new[]{"Western","Drama"}),
    (Title:"Parasite",        Director:"Bong",      Year:2019, Genres:new[]{"Drama","Thriller"}),
    (Title:"MemoriesOfMurder",Director:"Bong",      Year:2003, Genres:new[]{"Crime","Drama"}),
    (Title:"Avatar",          Director:"Cameron",   Year:2009, Genres:new[]{"SciFi","Action"}),
};

foreach (var f in films) {
    var filmNode = U(g, "ex:"+f.Title);
    var dirNode  = U(g, "ex:"+f.Director);
    g.Assert(filmNode, RDF_TYPE,    U(g, "ex:Film"));
    g.Assert(dirNode,  RDF_TYPE,    U(g, "ex:Director"));
    g.Assert(filmNode, DIRECTED_BY, dirNode);
    g.Assert(filmNode, RELEASED,    LitInt(g, f.Year));
    foreach (var gn in f.Genres) {
        var gNode = U(g, "ex:"+gn);
        g.Assert(gNode,   RDF_TYPE,  U(g, "ex:Genre"));
        g.Assert(filmNode, HAS_GENRE, gNode);
    }
}
Console.WriteLine($"KG cinema : {g.Triples.Count} triplets assertes.");

KG cinema : 54 triplets assertes.


**Interprétation** : chaque ligne de la table source engendre plusieurs triplets (type, directedBy, releasedIn, hasGenre × nombre de genres). Le KG est maintenant un graphe RDF interrogable. Le compte exact de triplets reflète la taille du domaine — on le retrouve dans le twin Python.

### 2.3 Sérialisation Turtle

Le KG peut être sérialisé dans plusieurs formats RDF. Turtle (`.ttl`) est le format texte lisible.

In [4]:
var fmt = new TurtleFormatter(g);
foreach (var t in g.Triples.Take(6))
    Console.WriteLine(fmt.Format(t));

ex:Director a ex:Class .


ex:Film a ex:Class .


ex:Genre a ex:Class .


ex:Inception a ex:Film .


ex:Nolan a ex:Director .


ex:Inception ex:directedBy ex:Nolan .


## 3. Interrogation SPARQL

SPARQL est le langage de requête standard du W3C pour RDF. On interroge le KG par **motifs graphe** (graph patterns).

### 3.1 Films d'un réalisateur donné

In [5]:
string q1 = @"
PREFIX ex: <http://example.org/cinema#>
SELECT ?title ?year WHERE {
    ?title ex:directedBy ex:Nolan ;
           ex:releasedIn ?year .
} ORDER BY DESC(?year)";
var results = (SparqlResultSet)g.ExecuteQuery(q1);
Console.WriteLine($"Films de Nolan (ordre annee desc) : {results.Count}");
foreach (SparqlResult r in results)
    Console.WriteLine($"  {r["title"].ToString().Replace("http://example.org/cinema#","")} ({((ILiteralNode)r["year"]).Value})");

Films de Nolan (ordre annee desc) : 3


  Dunkirk (2017)


  Interstellar (2014)


  Inception (2010)


### 3.2 Agrégats : nombre de films par genre

SPARQL supporte `GROUP BY` et `COUNT`, comme SQL.

In [6]:
string q2 = @"
PREFIX ex: <http://example.org/cinema#>
SELECT ?genre (COUNT(?film) AS ?n) WHERE {
    ?film ex:hasGenre ?genre .
} GROUP BY ?genre ORDER BY DESC(?n)";
var res2 = (SparqlResultSet)g.ExecuteQuery(q2);
Console.WriteLine("Nombre de films par genre :");
foreach (SparqlResult r in res2)
    Console.WriteLine($"  {r["genre"].ToString().Replace("http://example.org/cinema#","")} : {((ILiteralNode)r["n"]).Value}");

Nombre de films par genre :


  Drama : 6


  SciFi : 3


  Action : 2


  Crime : 2


  War : 1


  Western : 1


  Thriller : 1


**Interprétation** : SPARQL permet de projeter, filtrer, agréger — exactement comme SQL, mais sur un graphe RDF. Les comptes (Drama, SciFi, Crime…) concordent avec la table source : Drama apparaît 6 fois (Inception non, Interstellar oui, Dunkirk oui, PulpFiction oui, Django oui, Parasite oui, MemoriesOfMurder oui → vérifiez à la main). On retrouve ces nombres dans le twin Python.

## 4. Adjacence et parcours de graphe

En Python, `networkx` fournit `DiGraph` et BFS/DFS. En C# sans librairie externe, on construit l'**adjacence** à la main (dictionnaire `Dictionary<INode, List<INode>>`) et un **parcours en largeur** (BFS). C'est l'occasion de comprendre ce que networkx fait sous le capot.

In [7]:
// Construction de la liste d'adjacence : noeud -> voisins sortants
var adj = new Dictionary<INode, List<INode>>();
foreach (var t in g.Triples) {
    if (!adj.ContainsKey(t.Subject)) adj[t.Subject] = new List<INode>();
    adj[t.Subject].Add(t.Object);
}
Console.WriteLine($"Adjacence : {adj.Count} noeuds avec au moins 1 arete sortante.");

// BFS (static, params explicites -- evite la capture de variables top-level)
static List<INode> Bfs(Dictionary<INode,List<INode>> adj, INode start, int maxDepth) {
    var visited = new HashSet<INode>{ start };
    var frontier = new Queue<(INode n, int d)>();
    frontier.Enqueue((start, 0));
    var order = new List<INode>{ start };
    while (frontier.Count > 0) {
        var (cur, d) = frontier.Dequeue();
        if (d >= maxDepth) continue;
        if (!adj.ContainsKey(cur)) continue;
        foreach (var nb in adj[cur])
            if (visited.Add(nb)) { order.Add(nb); frontier.Enqueue((nb, d+1)); }
    }
    return order;
}
var reach = Bfs(adj, U(g, "ex:Inception"), 2);
Console.WriteLine($"BFS depuis Inception (profondeur<=2) : {reach.Count} noeuds atteints.");

Adjacence : 22 noeuds avec au moins 1 arete sortante.


BFS depuis Inception (profondeur<=2) : 9 noeuds atteints.


**Interprétation** : `Inception` → `Nolan` (directedBy), `2010` (releasedIn), `SciFi`/`Action` (hasGenre), puis depuis `Nolan` → autres films Nolan, etc. Le BFS explore le voisinage en éventail, exactement comme `networkx.single_source_shortest_path_length`. La profondeur 2 atteint le co-voisinage direct.

### 4.1 Visualisation ASCII du sous-graphe

Le twin Python utilise `matplotlib` + `pyvis` (interactif web). En C# sans dépendance graphique, on rend le sous-graphe en **ASCII** — substitution documentée (cf verdict SOTA en conclusion).

In [8]:
static string Short(INode n) {
    var s = n.ToString();
    return s.Replace("http://example.org/cinema#","")
            .Replace("http://www.w3.org/1999/02/22-rdf-syntax-ns#","rdf:")
            .Replace("http://www.w3.org/2000/01/rdf-schema#","rdfs:");
}
static string Pred(IUriNode p, IUriNode DIRECTED_BY, IUriNode HAS_GENRE, IUriNode RELEASED, IUriNode RDF_TYPE) {
    if (p.Equals(DIRECTED_BY)) return "--directedBy-->";
    if (p.Equals(HAS_GENRE))   return "--hasGenre-->";
    if (p.Equals(RELEASED))    return "--releasedIn-->";
    if (p.Equals(RDF_TYPE))    return "--a-->";
    return "--"+Short(p)+"-->";
}
var center = U(g, "ex:Inception");
Console.WriteLine($"Sous-graphe autour de {Short(center)} :");
foreach (var t in g.Triples.Where(t => t.Subject.Equals(center) || t.Object.Equals(center))) {
    var obj = t.Object is ILiteralNode lit ? lit.Value : Short(t.Object);
    Console.WriteLine($"  {Short(t.Subject)} {Pred((IUriNode)t.Predicate, DIRECTED_BY, HAS_GENRE, RELEASED, RDF_TYPE)} {obj}");
}

Sous-graphe autour de Inception :


  Inception --a--> Film


  Inception --directedBy--> Nolan


  Inception --releasedIn--> 2010


  Inception --hasGenre--> SciFi


  Inception --hasGenre--> Action


## 5. Qualité et validation d'un Knowledge Graph

Un KG réel doit être **validé** : pas de nœuds orphelins (sans aucune relation), complétude (toutes les entités attendues présentes), cohérence (pas de contradictions). On implémente 3 vérifications from-scratch.

In [9]:
// 5.1 Classes declarees mais jamais instanciees (orphelines au sens large)
var classes = g.Triples.Where(t => t.Predicate.Equals(RDF_TYPE) && t.Object.Equals(U(g,"ex:Class")))
                      .Select(t=>t.Subject).ToList();
var orphans = classes.Where(c => !g.Triples.Any(t => t.Subject.Equals(c) && !t.Predicate.Equals(RDF_TYPE))).ToList();
Console.WriteLine($"Classes declarees : {classes.Count}, orphelines : {orphans.Count}");

// 5.2 Completude : chaque Film a directeur + annee + au moins 1 genre ?
var allFilms = g.Triples.Where(t => t.Predicate.Equals(RDF_TYPE) && t.Object.Equals(U(g,"ex:Film")))
                       .Select(t=>t.Subject).ToList();
int complete = 0;
foreach (var f in allFilms) {
    bool hasDir   = g.Triples.Any(t => t.Subject.Equals(f) && t.Predicate.Equals(DIRECTED_BY));
    bool hasYear  = g.Triples.Any(t => t.Subject.Equals(f) && t.Predicate.Equals(RELEASED));
    bool hasGenre = g.Triples.Any(t => t.Subject.Equals(f) && t.Predicate.Equals(HAS_GENRE));
    if (hasDir && hasYear && hasGenre) complete++;
}
Console.WriteLine($"Films complets (directeur+annee+genre) : {complete}/{allFilms.Count}");

// 5.3 Distribution des annees
var years = g.Triples.Where(t => t.Predicate.Equals(RELEASED))
    .Select(t => int.Parse(((ILiteralNode)t.Object).Value)).OrderBy(y=>y).ToList();
Console.WriteLine($"Annees : min={years.First()}, max={years.Last()}, mediane={years[years.Count/2]}");

Classes declarees : 3, orphelines : 3


Films complets (directeur+annee+genre) : 8/8


Annees : min=1994, max=2019, mediane=2012


**Interprétation** : les 3 métriques de qualité valident que le KG est bien formé. (1) **Complétude** : `complete == allFilms.Count` (8/8) confirme qu'aucun film n'a de champ manquant (directeur, année, genre). (2) **Cohérence du range** : les années s'étalent de 1994 à 2019, médiane 2012 — une plage réaliste pour un corpus de cinéma moderne. (3) **Orphelins** : les 3 classes « orphelines » détectées (`Director`, `Film`, `Genre`) sont les **méta-types** eux-mêmes — elles n'apparaissent comme sujet que de leur propre `rdf:type ex:Class`, ce qui est **attendu** (ce sont les types, pas des instances) et non un défaut de données. Ces vérifications sont l'équivalent C# du bloc « Qualité » du twin Python.

## 6. Centralité : degré des réalisateurs

Le twin Python calcule un **PageRank** (networkx). PageRank complet nécessite une itération matricielle ; ici on commence par la métrique la plus simple et robuste — le **degré sortant** (nombre de films dirigés) — qui est la base sur laquelle PageRank se construit.

In [10]:
var deg = new Dictionary<string,int>();
foreach (var t in g.Triples.Where(t => t.Predicate.Equals(DIRECTED_BY))) {
    var dir = Short(t.Object);
    deg[dir] = deg.GetValueOrDefault(dir, 0) + 1;
}
var ranked = deg.OrderByDescending(kv => kv.Value).ThenBy(kv => kv.Key).ToList();
Console.WriteLine("Classement des realisateurs par productivite (nb films) :");
for (int i = 0; i < ranked.Count; i++)
    Console.WriteLine($"  {i+1}. {ranked[i].Key} : {ranked[i].Value} films");

Classement des realisateurs par productivite (nb films) :


  1. Nolan : 3 films


  2. Bong : 2 films


  3. Tarantino : 2 films


  4. Cameron : 1 films


**Interprétation** : le degré sortant (`directedBy` inversé) classe les réalisateurs par productivité dans le KG. Nolan mène (3 films), suivi de Bong et Tarantino (2 chacun), puis Cameron (1). C'est le **degree centrality** — la métrique de centralité la plus directe. Le PageRank du twin Python raffine en pondérant par l'importance des voisins, mais le classement brut par degré est déjà une excellente approximation sur un petit graphe.

## 7. Verdict SOTA — ce qui est couvert et ce qui ne l'est pas

| Capacité | dotNetRDF (this twin) | Twin Python | Verdict |
|----------|----------------------|-------------|---------|
| RDF triplets, namespaces, sérialisation | OK `Graph`, `Assert`, TurtleFormatter | rdflib | **SOTA-OK** |
| SPARQL (SELECT, GROUP BY, COUNT) | OK `ExecuteQuery` moteur complet | rdflib | **SOTA-OK** |
| Adjacence, BFS, centralité degré | OK from-scratch (dictionnaire + queue) | networkx | **SOTA-OK** (réimplémenté) |
| Visualisation graphe | ASCII (sous-graphe) | matplotlib + **pyvis** (interactif web) | **INTRINSIC** — pyvis produit du HTML/JS interactif (drag/zoom), non reproductible en ASCII ; matplotlib → ASCII documenté |
| Couche d'abstraction KG | dotNetRDF direct | **kglab** | **INTRINSIC** — kglab est une surcouche Python (assistant sklearn-like), pas d'équivalent .NET idiomatique ; on utilise dotNetRDF directement (équivalent fonctionnel) |
| Raisonnement OWL (HermiT) | `IOwlReasoner` interface (pas d'impl native) | owlready2 + HermiT (JVM) | **RECOVERABLE-MACHINE** — HermiT/Pellet tournent sur JVM (Java) ; SW-13 documente déjà cet écart ; dotNetRDF expose l'interface mais pas d'implémentation OWL 2 DL native |

**Conclusion honnête** : ce twin couvre intégralement le cœur pédagogique (construction RDF, SPARQL, parcours graphe, qualité, centralité) avec le **vrai moteur dotNetRDF** (pas une réimplémentation jouet). Les écarts (pyvis interactif, kglab, HermiT) sont **intrinsèques ou recoverable-machine** et documentés en prose, pas maquillés.

## Exemples guidés

Les cellules ci-dessous montrent des requêtes étendues sur le KG.

In [11]:
// Exemple guide 1 : paires de realisateurs partageant des genres
string q3 = @"
PREFIX ex: <http://example.org/cinema#>
SELECT ?d1 ?d2 (COUNT(DISTINCT ?g) AS ?shared) WHERE {
    ?f1 ex:directedBy ?d1 ; ex:hasGenre ?g .
    ?f2 ex:directedBy ?d2 ; ex:hasGenre ?g .
    FILTER(STR(?d1) < STR(?d2))
} GROUP BY ?d1 ?d2 ORDER BY DESC(?shared)";
var res3 = (SparqlResultSet)g.ExecuteQuery(q3);
Console.WriteLine("Paires de realisateurs partageant des genres :");
foreach (SparqlResult r in res3)
    Console.WriteLine($"  {Short(r["d1"])} & {Short(r["d2"])} : {((ILiteralNode)r["shared"]).Value} genre(s) commun(s)");

Paires de realisateurs partageant des genres :


  Cameron & Nolan : 2 genre(s) commun(s)


  Bong & Tarantino : 2 genre(s) commun(s)


  Bong & Nolan : 1 genre(s) commun(s)


  Nolan & Tarantino : 1 genre(s) commun(s)


In [12]:
// Exemple guide 2 : filmographie de Bong avec genres agreges
// En C# verbatim string (@"..."), un " litteral s'ecrit "" (double quote)
string q4 = @"
PREFIX ex: <http://example.org/cinema#>
SELECT ?film (GROUP_CONCAT(DISTINCT ?g; SEPARATOR="", "") AS ?genres) WHERE {
    ?film ex:directedBy ex:Bong ; ex:hasGenre ?g .
} GROUP BY ?film ORDER BY ?film";
var res4 = (SparqlResultSet)g.ExecuteQuery(q4);
Console.WriteLine("Filmographie de Bong (genres agreges) :");
foreach (SparqlResult r in res4)
    Console.WriteLine($"  {Short(r["film"])} : {((ILiteralNode)r["genres"]).Value.Replace("http://example.org/cinema#","")}");

Filmographie de Bong (genres agreges) :


  MemoriesOfMurder : Crime, Drama


  Parasite : Drama, Thriller


## Exercices à compléter

Les exercices ci-dessous sont à compléter. Les stubs s'exécutent sans erreur (C.1) — remplacez `null` / le TODO par votre code.

### Exercice 1 : Films d'une année donnée

Écrivez une requête SPARQL qui retourne tous les films sortis **après 2010**, triés par année. *Indice : `FILTER(?year > 2010)`.*

In [13]:
// Exercice 1 : a completer
// string q = "PREFIX ex: <http://example.org/cinema#> SELECT ?film ?year WHERE { ?film ex:releasedIn ?year . FILTER(?year > 2010) } ORDER BY ?year";
// var res = (SparqlResultSet)g.ExecuteQuery(q);
// foreach (SparqlResult r in res) Console.WriteLine($"  {Short(r["film"])} ({r["year"]})");
Console.WriteLine("Exercice 1 a completer");

Exercice 1 a completer


### Exercice 2 : Degré entrant d'un genre

Comptez combien de films référencent chaque genre, **sans SPARQL** (en parcourant `g.Triples` directement en C#). *Indice : filtrez `t.Predicate.Equals(HAS_GENRE)` puis groupez par `t.Object`.*

In [14]:
// Exercice 2 : a completer
// var genreCount = new Dictionary<string,int>();
// foreach (var t in g.Triples.Where(t => t.Predicate.Equals(HAS_GENRE))) { ... }
Console.WriteLine("Exercice 2 a completer");

Exercice 2 a completer


### Exercice 3 : Chemin le plus court entre deux entités

Utilisez le BFS fourni pour trouver le **chemin le plus court** (nombre d'arêtes) entre `Inception` et `PulpFiction` (via un réalisateur ou genre commun). *Indice : relancez `Bfs(adj, U(g,"ex:Inception"), 3)` et notez si `U(g,"ex:PulpFiction")` est dans le résultat.*

In [15]:
// Exercice 3 : a completer
// var path = Bfs(adj, U(g, "ex:Inception"), 3);
// if (path.Contains(U(g, "ex:PulpFiction"))) Console.WriteLine("PulpFiction atteint depuis Inception");
Console.WriteLine("Exercice 3 a completer");

Exercice 3 a completer


## Résumé et perspectives

Ce twin C# a couvert l'arc complet de la gestion d'un **Knowledge Graph** avec dotNetRDF :

1. **Construction** : définition d'un vocabulaire RDF, transformation de données structurées en triplets.
2. **Interrogation** : SPARQL (SELECT, agrégats, filtres, GROUP_CONCAT).
3. **Parcours** : adjacence from-scratch, BFS, centralité de degré.
4. **Qualité** : détection d'orphelins, complétude, cohérence.
5. **Visualisation** : ASCII (sous-graphe), avec les limites documentées vs pyvis/matplotlib.

**Perspectives** : pour un KG plus large (DBpedia, Wikidata), dotNetRDF supporte les **endpoints SPARQL distants** (`SparqlRemoteEndpoint`) — on pourrait interroger `https://dbpedia.org/sparql` directement. Le raisonnement OWL complet (HermiT) reste un pont vers la JVM, déjà documenté dans SW-13.